### Описание

Цель: обучить gemma-3-1b-it решать задачу Countdown

Задачи:
1) Создать датасет
2) Обучить модель
3) Создать сабмить

Как решалась задача:
Часть обучающей выборки была взята с verified_Qwen3-4B-Instruct-2507
Но поскольку в ней были примеры лишь на 3 и 4 значения, то дополнительно было сигенерировано 10к данных, которые замешались в 27к примеров готового верефицированного датасета.
Данные приведены к формату messages (System, User, Assistant).
Так мы создали датасеты в 37к значений.

Далее мы загрузили модель(Использована квантованная (4-bit) версия). Применен метод LoRA.
Включен weight_decay (0.01) и градиентный клиппинг (1.0) для предотвращения взрыва градиентов.
Модель обучалась использовать теги <think> для промежуточных вычислений
Подабрали параметры обучения, которые позволяли обучаться модель на гугл колаб (прописаны в конфиг - БЛОК 2)

Потом обучили модель, создали сабмит и выгрузили на кагл

### Блок 1: Установка и импорт библиотек

In [ ]:
# %%
"""Установка зависимостей"""
!pip install -q unsloth
!pip install -q transformers datasets accelerate bitsandbytes peft trl
!pip install -q xformers --index-url https://download.pytorch.org/whl/cu121


In [ ]:
!pip install --upgrade trl==0.8.6

In [ ]:
!pip install --upgrade trl==0.8.6 peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# без этого не сработает

import os
import shutil

os.environ["WANDB_DISABLED"] = "true"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# удаляем все старые сломанные protobuf-файлы wandb
wandb_dir = "/usr/local/lib/python3.12/dist-packages/wandb"
if os.path.exists(wandb_dir):
    shutil.rmtree(wandb_dir, ignore_errors=True)

print("Старые файлы wandb удалены + wandb отключён")

In [ ]:
"""Импорт библиотек"""
import os
import re
import ast
import json
import time
import random
import operator
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from itertools import permutations, product

import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset, concatenate_datasets
from huggingface_hub import login
from transformers import TrainingArguments, Trainer

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)

from peft import LoraConfig, PeftModel, PeftConfig
from trl import SFTTrainer, SFTConfig

from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only

print("Все библиотеки загружены")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Память: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

### Блок 2: Конфигурация и авторизация

Обратим внимание, что была загружена сеть twm-opensource/gemma-3-1b-it". В условиях не было огаварено, что нельзя использовать вариации, так что  была загружена модель, которая обучалась на других математических задачках(не похожих на то, что решаем сейчас)

In [ ]:
"""Конфигурация обучения"""
CFG = {
    # Модель
    "model_name": "google/gemma-3-1b-it",
    "hf_hub_model": "ralifgrannik/gemma-3-1b-countdown-sft",
    "hf_token": "", # Убрал, нужно заполнить <-<-<--------------------------<-<-<-<-<-<-<-<-<-<-<-<---------------------------<-<-<-<-<-<-<-<-<-----------------

    # Данные
    "dataset_name": "HuggingFaceTB/Countdown-Task-GOLD",
    "dataset_config": "verified_Qwen3-4B-Instruct-2507",
    "val_split": 0.05,
    "synth_5_6_count": 2000,

    # Обучение
    "max_seq_length": 768,
    "learning_rate": 2e-4,
    "batch_size": 4,
    "grad_accum": 4,
    "num_epochs": 2,
    "warmup_ratio": 0.05,
    "lr_scheduler": "cosine",
    "optim": "adamw_8bit",
    "max_grad_norm": 1.0,
    "weight_decay": 0.01,

    # LoRA
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0,

    # Сохранение
    "eval_steps": 200,
    "save_steps": 200,
    "save_total_limit": 3,
    "logging_steps": 10,
    "load_best": True,

    # Инференс
    "max_new_tokens": 1024,
    "batch_inf": 8,
    "test_csv": "test_public.csv",

    # Пути
    "output_dir": "./gemma-countdown-sft/checkpoints",
    "submission_csv": "./submission.csv",
    "seed": 42,
}

# Создаём директории
Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)

# Фиксируем seed
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])

print("Конфигурация:")
print(json.dumps(CFG, indent=2))

# %%
"""Авторизация в HuggingFace"""
try:
    login(token=CFG["hf_token"], add_to_git_credential=True)
    print("HuggingFace авторизован")
except:
    print("Авторизация не удалась, продолжаем без неё")

### Блок 3: Обработка датасета

#### Загрузка верифицированного датасета + генерация новых примеров

In [ ]:
"""Подготовка сбалансированного датасета из 15k примеров"""
import random
import operator
import re
from collections import Counter
from datasets import load_dataset, Dataset, concatenate_datasets

def generate_synthetic_examples(n_examples, n_nums_range=(5, 6)):
    """Быстрый генератор синтетики"""
    SYSTEM_PROMPT = (
        "You are a helpful assistant. You first think about the reasoning "
        "process in the mind and then provide the user with the answer."
    )

    ops = {'+': operator.add, '-': operator.sub,
           '*': operator.mul, '/': operator.truediv}

    examples = []
    pbar = 0

    while len(examples) < n_examples:
        n = random.randint(*n_nums_range)
        original_nums = random.choices(range(1, 101), k=n)

        current_elements = [{"val": float(x), "expr": str(x), "is_complex": False}
                           for x in original_nums]
        temp_elements = list(current_elements)
        random.shuffle(temp_elements)

        while len(temp_elements) > 1:
            a = temp_elements.pop()
            b = temp_elements.pop()
            op_sym = random.choice(list(ops.keys()))

            if op_sym == '/' and (b['val'] == 0 or a['val'] % b['val'] != 0):
                op_sym = random.choice(['+', '-', '*'])
            if op_sym == '-' and a['val'] < b['val']:
                a, b = b, a

            res_val = ops[op_sym](a['val'], b['val'])

            a_str = f"({a['expr']})" if a['is_complex'] and op_sym in '*/' else a['expr']
            b_str = f"({b['expr']})" if b['is_complex'] or (op_sym == '-' and b['is_complex']) else b['expr']

            new_expr = f"{a_str} {op_sym} {b_str}"
            temp_elements.append({"val": res_val, "expr": new_expr, "is_complex": True})

        target = int(temp_elements[0]['val'])
        solution = temp_elements[0]['expr']

        if target <= 0 or target > 2000:
            continue

        user_content = (
            f"Using the numbers {original_nums}, create an equation that equals {target}. "
            f"You can use basic arithmetic operations (+, -, *, /) and each number "
            f"can only be used once. Show your work in <think> </think> tags. "
            f"And return the final equation and answer in <answer> </answer> tags."
        )

        asst_content = (
            f"<think>\nLet me find a solution using {original_nums} to reach {target}.\n\n"
            f"After trying different combinations:\n{solution} = {target} ✓\n</think>\n"
            f"<answer>\n{solution} = {target}\n</answer>"
        )

        examples.append({
            "target": target,
            "nums": original_nums,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": asst_content},
            ]
        })

        if len(examples) % 500 == 0 and len(examples) > pbar:
            pbar = len(examples)
            print(f"  Сгенерировано: {len(examples)}/{n_examples}")

    print(f"Синтетические данные готовы: {len(examples)} примеров")
    return examples


def validate_equation(eq_str, nums, target):
    try:
        allowed = set("0123456789 +-*/().")
        if not set(eq_str) <= allowed:
            return False
        nums_in_eq = [int(x) for x in re.findall(r'\d+', eq_str)]
        if sorted(nums_in_eq) != sorted(nums):
            return False
        return abs(eval(eq_str) - target) < 1e-6
    except:
        return False


def extract_equation(assistant_content):
    m = re.search(r'<answer>(.*?)</answer>', assistant_content, re.DOTALL)
    if not m:
        return None
    raw = m.group(1).strip()
    return raw.split('=')[0].strip() if '=' in raw else raw


def load_and_prepare_dataset(cfg, total_train_size=15000):
    """
    Загружает датасет, добавляет синтетические данные и возвращает total_train_size примеров
    """
    print(f"\n📥 Загружаем {cfg['dataset_name']}...")
    ds = load_dataset(cfg["dataset_name"], cfg["dataset_config"], split="train")
    print(f"  Загружено: {len(ds)} примеров")

    def is_valid(example):
        asst = next((m for m in example["messages"] if m["role"] == "assistant"), None)
        if not asst:
            return False
        eq = extract_equation(asst["content"])
        return validate_equation(eq, example["nums"], example["target"]) if eq else False

    print("🔍 Фильтруем...")
    ds_filtered = ds.filter(is_valid)
    print(f"  После фильтрации: {len(ds_filtered)}")

    print("Дедупликация...")
    seen = set()
    keep_idx = []
    for i, ex in enumerate(ds_filtered):
        key = (ex["target"], tuple(sorted(ex["nums"])))
        if key not in seen:
            seen.add(key)
            keep_idx.append(i)
    ds_deduped = ds_filtered.select(keep_idx)
    print(f"  После дедупликации: {len(ds_deduped)}")

    # Статистика по числам в оригинальном датасете
    print("\nОригинальный датасет:")
    nums_dist = Counter(len(ex["nums"]) for ex in ds_deduped)
    for k in sorted(nums_dist):
        print(f"    {k} чисел: {nums_dist[k]} ({nums_dist[k]/len(ds_deduped)*100:.1f}%)")

    # Генерируем синтетику (10000 примеров с 5-6 числами)
    synth_count = 10000
    print(f"\nГенерация {synth_count} синтетических примеров (5-6 чисел)...")
    synth_list = generate_synthetic_examples(synth_count)
    ds_synth = Dataset.from_list(synth_list)

    # Объединяем
    ds_combined = concatenate_datasets([ds_deduped, ds_synth])
    ds_combined = ds_combined.shuffle(seed=cfg["seed"])

    print(f"\nОбъединённый датасет: {len(ds_combined)} примеров")
    nums_dist_combined = Counter(len(ex["nums"]) for ex in ds_combined)
    for k in sorted(nums_dist_combined):
        print(f"    {k} чисел: {nums_dist_combined[k]} ({nums_dist_combined[k]/len(ds_combined)*100:.1f}%)")

    # Берём ровно total_train_size примеров для train
    if len(ds_combined) > total_train_size + int(total_train_size * cfg["val_split"]):
        ds_combined = ds_combined.select(range(total_train_size + int(total_train_size * cfg["val_split"])))

    # Разделяем на train/val
    val_size = int(len(ds_combined) * cfg["val_split"])
    split = ds_combined.train_test_split(test_size=val_size, seed=cfg["seed"])

    ds_train = split["train"]
    ds_val = split["test"]

    if len(ds_train) > total_train_size:
        ds_train = ds_train.select(range(total_train_size))

    print(f"\nИтоговый датасет:")
    print(f"  Train: {len(ds_train)} примеров")
    print(f"  Val:   {len(ds_val)} примеров")

    # Финальная статистика train
    print("\nTrain распределение:")
    train_dist = Counter(len(ex["nums"]) for ex in ds_train)
    for k in sorted(train_dist):
        print(f"    {k} чисел: {train_dist[k]} ({train_dist[k]/len(ds_train)*100:.1f}%)")

    return ds_train, ds_val


# Обновляем CFG
CFG["synth_5_6_count"] = 10000  # 10k синтетики
CFG["val_split"] = 0.05

# Загружаем датасет
ds_train, ds_val = load_and_prepare_dataset(CFG, total_train_size=37000)

print("\nДатасет готов!")

#### Загрузка модели и токенизатора:

In [ ]:
"""Загрузка модели через стандартный transformers (БЕЗ Unsloth)"""
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"\nЗагружаем {CFG['model_name']}...")

# Конфигурация 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Загружаем базовую модель
model = AutoModelForCausalLM.from_pretrained(
    CFG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",  # ВАЖНО: отключаем flash attention
    token=CFG.get("hf_token"),
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# Загружаем токенизатор
tokenizer = AutoTokenizer.from_pretrained(
    CFG["model_name"],
    token=CFG.get("hf_token"),
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Применяем LoRA
lora_config = LoraConfig(
    r=CFG["lora_r"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Модель загружена. LoRA trainable: {trainable:,} ({trainable/total*100:.2f}%)")

#### Формирование датасета:

In [ ]:
def format_dataset(dataset, tokenizer, max_length):
    formatted_texts = []
    for example in dataset:
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        formatted_texts.append(text)

    tokenized = tokenizer(
        formatted_texts,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    tokenized["labels"] = tokenized["input_ids"].clone()

    # Маскируем промпт
    response_template = "<start_of_turn>model\n"
    response_tokens = tokenizer.encode(response_template, add_special_tokens=False)

    for i in range(len(tokenized["input_ids"])):
        input_ids = tokenized["input_ids"][i].tolist()
        for j in range(len(input_ids) - len(response_tokens)):
            if input_ids[j:j+len(response_tokens)] == response_tokens:
                tokenized["labels"][i, :j+len(response_tokens)] = -100
                break

    return tokenized


class CountdownDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.input_ids = data["input_ids"]
        self.attention_mask = data["attention_mask"]
        self.labels = data["labels"]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


print("📝 Форматируем датасет...")
train_data = format_dataset(ds_train, tokenizer, CFG["max_seq_length"])
val_data = format_dataset(ds_val, tokenizer, CFG["max_seq_length"])

train_dataset = CountdownDataset(train_data)
val_dataset = CountdownDataset(val_data)

print(f"✅ Train: {len(train_dataset)}, Val: {len(val_dataset)}")

### Блок 4: Обучение

In [ ]:
from transformers import Trainer, TrainingArguments
import time
import torch

torch.cuda.empty_cache()

CFG["batch_size"] = 2
CFG["grad_accum"] = 4

steps_per_epoch = len(train_dataset) // (CFG["batch_size"] * CFG["grad_accum"])
warmup_steps = int(CFG["warmup_ratio"] * steps_per_epoch * CFG["num_epochs"])

print(f"\n📊 Параметры обучения:")
print(f"  Train: {len(train_dataset)} | Batch: {CFG['batch_size']}x{CFG['grad_accum']}={CFG['batch_size']*CFG['grad_accum']}")
print(f"  Шагов в эпохе: {steps_per_epoch} | Всего: {steps_per_epoch * CFG['num_epochs']}")

training_args = TrainingArguments(
    output_dir=CFG["output_dir"],
    num_train_epochs=CFG["num_epochs"],
    per_device_train_batch_size=CFG["batch_size"],
    per_device_eval_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["learning_rate"],
    optim="paged_adamw_8bit",
    weight_decay=CFG["weight_decay"],
    max_grad_norm=CFG["max_grad_norm"],
    warmup_steps=warmup_steps,
    lr_scheduler_type=CFG["lr_scheduler"],
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=CFG["eval_steps"],
    save_strategy="steps",
    save_steps=CFG["save_steps"],
    save_total_limit=CFG["save_total_limit"],
    load_best_model_at_end=CFG["load_best"],
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=CFG["logging_steps"],
    report_to="none",
    dataloader_num_workers=0,
    seed=CFG["seed"],
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print("\n🚀 Начинаем обучение...\n")
start_time = time.time()

trainer.train()

print(f"\n✅ Завершено за {(time.time()-start_time)/60:.1f} минут")

# Сохраняем
model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")
print("✅ Модель сохранена")

### Блок 5: Тест (инференс)

In [ ]:
import pandas as pd
import ast
import re
import time
import torch


model.eval()
torch.cuda.empty_cache()

def extract_equation(text):
    """Извлекает уравнение из <answer> тега"""
    m = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if not m:
        return None
    raw = m.group(1).strip()
    if '=' in raw:
        return raw.split('=')[0].strip()
    return raw

def simple_fallback(nums, target):
    """Простой fallback если модель не выдала ответ"""
    if len(nums) >= 2:
        return f"{nums[0]}+{nums[1]}"
    return "0"

# Загружаем тестовые данные
print(f"\n📥 Загружаем тестовые данные: {CFG['test_csv']}")
test_df = pd.read_csv(CFG["test_csv"])
test_df["nums"] = test_df["nums"].apply(ast.literal_eval)
print(f"   Тестовых примеров: {len(test_df)}")

SYSTEM_PROMPT = "You are a helpful math assistant. Answer only with the equation in <answer> tags."

equations = []
n_fallback = 0
batch_size = CFG["batch_inf"]

print(f"\n⏳ Генерируем ответы (batch_size={batch_size})...")
start_time = time.time()

for batch_start in range(0, len(test_df), batch_size):
    batch = test_df.iloc[batch_start:batch_start + batch_size]

    prompts = []
    for _, row in batch.iterrows():
        nums = row["nums"]
        target = row["target"]

        user_content = (
            f"Using the numbers {nums}, create an equation that equals {target}. "
            f"Use +, -, *, / and each number exactly once. "
            f"Return the equation in <answer>equation = {target}</answer>."
        )

        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]

        prompt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
        prompts.append(prompt)

    # Токенизируем батч
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=CFG["max_seq_length"],
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CFG["max_new_tokens"],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Декодируем
    input_len = inputs.input_ids.shape[1]
    for i, (out, (_, row)) in enumerate(zip(outputs, batch.iterrows())):
        generated = tokenizer.decode(
            out[input_len:],
            skip_special_tokens=True
        )

        eq = extract_equation(generated)

        if eq is None:
            eq = simple_fallback(row["nums"], row["target"])
            n_fallback += 1

        equations.append(eq)

    if (batch_start // batch_size) % 10 == 0:
        done = min(batch_start + batch_size, len(test_df))
        elapsed = time.time() - start_time
        speed = done / elapsed if elapsed > 0 else 0
        eta = (len(test_df) - done) / speed if speed > 0 else 0
        print(f"  [{done:>4}/{len(test_df)}] speed={speed:.1f} ex/s ETA={eta/60:.1f}m fallback={n_fallback}")

elapsed = time.time() - start_time
print(f"\n✅ Инференс завершён за {elapsed/60:.1f} минут")
print(f"   Fallback использован: {n_fallback} раз ({n_fallback/len(test_df)*100:.1f}%)")

# Сохраняем сабмит
submission = pd.DataFrame({
    "id": test_df["id"],
    "equation": equations
})
submission.to_csv(CFG["submission_csv"], index=False)
print(f"\n📁 Submission сохранён: {CFG['submission_csv']}")
print("\n📝 Первые 10 ответов:")
for i in range(min(10, len(submission))):
    print(f"  ID {submission.iloc[i]['id']}: {submission.iloc[i]['equation']}")